In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassificationModel
from datetime import datetime
import psycopg2

### Initialize Spark session


In [2]:
spark = SparkSession.builder \
    .appName("Prediction") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4") \
    .getOrCreate()

#### Load the pre-trained Random Forest model


In [3]:
model = RandomForestClassificationModel.load("zrandom_forest_model")

##### Define the schema for Kafka messages


In [4]:
schema = StructType([
    StructField("patient_id", LongType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", IntegerType(), True),
    StructField("chestpaintype", IntegerType(), True),
    StructField("restingBP", IntegerType(), True),
    StructField("cholestrol", IntegerType(), True),
    StructField("fastingbloodsugar", IntegerType(), True),
    StructField("restingECG", IntegerType(), True),
    StructField("maxheartrate", IntegerType(), True),
    StructField("exerciseangia", IntegerType(), True),
    StructField("oldpeak", DoubleType(), True),
    StructField("slope", IntegerType(), True),
])

##### Kafka stream setup


In [5]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "heart") \
    .option("startingOffsets", "latest") \
    .load()

parsed_stream = kafka_stream.selectExpr("CAST(value AS STRING)") \
    .select(from_json("value", schema).alias("data")) \
    .select("data.*")

##### Features for prediction


In [6]:
feature_columns = ["age", "gender", "chestpaintype", "restingBP", "cholestrol",
                   "fastingbloodsugar", "restingECG", "maxheartrate", "exerciseangia",
                   "oldpeak", "slope"]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")


### PostgreSQL settings


In [10]:

db_config = {
    "host": "localhost",
    "database": "Patients_DW",
    "user": "postgres",
    "password": "postgres",
    "port": 5432
}


#### Process each micro-batch


In [11]:

def process_batch(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    batch_df = assembler.transform(batch_df)
    predictions = model.transform(batch_df)

    rows = predictions.select("patient_id", *feature_columns, "prediction", "probability").collect()

    try:
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        now = datetime.now()
        date, day, month, year, hour = now.date(), now.day, now.month, now.year, now.hour
        day_of_week = now.weekday() + 1

        cursor.execute("""
            INSERT INTO dim_time (date, day, month, year, hour, day_of_week)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT DO NOTHING
            RETURNING time_id
        """, (date, day, month, year, hour, day_of_week))

        result = cursor.fetchone()
        time_id = result[0] if result else None
        if not time_id:
            cursor.execute("""
                SELECT time_id FROM dim_time
                WHERE date=%s AND day=%s AND month=%s AND year=%s AND hour=%s AND day_of_week=%s
            """, (date, day, month, year, hour, day_of_week))
            time_id = cursor.fetchone()[0]

        for row in rows:
            cursor.execute("""
                INSERT INTO dim_patient (
                    patient_id, age, gender, chest_pain_type,
                    fasting_sugar, resting_ecg, exercise_angina, slope
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (patient_id) DO NOTHING
            """, (
                row['patient_id'], row['age'], row['gender'], row['chestpaintype'],
                row['fastingbloodsugar'], row['restingECG'], row['exerciseangia'], row['slope']
            ))

            prob = row['probability']
            prediction_score = float(prob[1]) if row['prediction'] == 1 else float(prob[0])

            cursor.execute("""
                INSERT INTO fact_predictions (
                    patient_id, time_id, resting_bp, cholesterol,
                    max_heart_rate, oldpeak, prediction_label, prediction_score
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            """, (
                row['patient_id'], time_id, row['restingBP'], row['cholestrol'],
                row['maxheartrate'], row['oldpeak'], int(row['prediction']), prediction_score
            ))

        conn.commit()
        cursor.close()
        conn.close()
        print("Batch processed and inserted.")

    except Exception as e:
        print("Error inserting batch to PostgreSQL:", e)


In [ ]:

# Start streaming with foreachBatch
query = parsed_stream.writeStream \
    .foreachBatch(process_batch) \
    .start()

query.awaitTermination()
